# Baseline vs optimized demand-response experiment

This executed experiment measures the project's existing baseline and OR-Tools strategies on the reproducible sample dataset. It uses domain entities, `ComparisonService`, `PlanningService`, `PlanningAnalysisService`, and `ExplanationService`; it does not substitute mock plans or hard-code measured results.

The selected case is the generated overloaded transformer. The committed JSON is loaded and regenerated from its recorded seed and parameters before planning. Rerunning with the same sample and software should reproduce the decisions and measurements.


## Scenario definition

| Item | Experiment setting |
|---|---|
| Scenario | Generated overloaded neighborhood transformer |
| Dataset | `data/synthetic_demo.json`; 4 transformers, 16 buildings |
| Seed | 2026, read from dataset metadata |
| Selected transformer | Generated overloaded case, TX-02 |
| Planning horizon | The selected event's three-hour UTC window, 15-minute slots |
| Baseline strategy | `BaselineStrategy`, deterministic building-ID order |
| Optimized strategy | `OptimizedStrategy` with OR-Tools CP-SAT |
| Objective weights | Peak reduction 0.70, comfort 0.30, energy cost 0.25 |
| Peak-reduction target | Read from the generated demand-response event |

The sample includes occupancy patterns, but the current `PlanningContext` has no occupancy field. The strategies therefore do not use those patterns; this appears in the error analysis.


In [1]:
from pathlib import Path
import json, platform, sys
from dataclasses import replace
from statistics import mean
PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / 'backend' / 'app').is_dir() and (p / 'data' / 'synthetic_demo.json').is_file())
BACKEND_ROOT = PROJECT_ROOT / 'backend'
sys.path.insert(0, str(BACKEND_ROOT))
import ortools
import pandas as pd
from app.application.services.comparison_service import ComparisonService
from app.application.services.explanation_service import ExplanationService
from app.application.services.planning_analysis_service import PlanningAnalysisService
from app.application.services.planning_service import PlanningService
from app.domain.strategies.baseline_strategy import BaselineStrategy
from app.domain.strategies.optimized_strategy import OptimizedStrategy
from app.domain.strategies.strategy_interface import PlanningContext
from app.infrastructure.data_generation.synthetic import dataset_to_dict, generate_synthetic_dataset
from app.infrastructure.solver.ortools_solver import ORToolsSolver
SAMPLE_PATH = PROJECT_ROOT / 'data' / 'synthetic_demo.json'
sample_json = json.loads(SAMPLE_PATH.read_text(encoding='utf-8'))
generation = sample_json['generation']; parameters = generation['parameters']
dataset = generate_synthetic_dataset(seed=generation['seed'],
    building_count=parameters['building_count'], transformer_count=parameters['transformer_count'],
    occupants_per_building=parameters['occupants_per_building'],
    appliances_per_building=parameters['appliances_per_building'],
    intervals_per_profile=parameters['intervals_per_profile'])
assert dataset_to_dict(dataset) == sample_json, 'Sample does not match its recorded seed and parameters.'
EXPERIMENT_SCENARIO = 'overloaded'
transformer = next(x for x in dataset.transformers if dataset.scenario_by_transformer[x.id] == EXPERIMENT_SCENARIO)
dr_event = next(x for x in dataset.events if x.transformer_id == transformer.id)
connected_building_ids = set(transformer.connected_building_ids)
buildings = tuple(x for x in dataset.buildings if x.id in connected_building_ids)
occupants = tuple(x for x in dataset.occupants if x.building_id in connected_building_ids)
appliances = tuple(x for x in dataset.appliances if x.building_id in connected_building_ids)
tariff = next(x for x in dataset.tariffs if x.id == f'TAR-{transformer.id[-2:]}')
transformer_profile = next(x for x in dataset.load_profiles if x.entity_id == transformer.id)
OBJECTIVE_WEIGHTS = {'peak_reduction': 0.7, 'comfort': 0.3, 'energy_cost': 0.25}
context = PlanningContext(dr_event, transformer, buildings, occupants, dataset.comfort_ranges,
    tariff, OBJECTIVE_WEIGHTS, appliances, transformer_load_profile=transformer_profile)
class RecordingSolver(ORToolsSolver):
    # Capture the actual solver result without altering solver behavior.
    def __init__(self): self.last_outcome = None
    def solve_problem(self, problem):
        self.last_outcome = super().solve_problem(problem)
        return self.last_outcome
recording_solver = RecordingSolver()
comparison = ComparisonService(BaselineStrategy(), OptimizedStrategy(recording_solver)).compare(context)
assert comparison.metrics is not None
baseline_decisions = list(comparison.first_decisions)
optimized_decisions = list(comparison.second_decisions)
analysis_service = PlanningAnalysisService()
baseline_metrics = analysis_service.analyze(context, baseline_decisions)
optimized_metrics = comparison.metrics
solver_status = recording_solver.last_outcome.status
uncontrolled_peak_kw = optimized_metrics.baseline_peak_load_kw
measured_optimized_peak_kw = optimized_metrics.optimized_peak_load_kw
measured_reduction_kw = optimized_metrics.peak_reduction_kw
target_reduction_kw = dr_event.target_reduction_kw
expected_optimized_peak_kw = uncontrolled_peak_kw - target_reduction_kw
target_shortfall_kw = max(0.0, target_reduction_kw - measured_reduction_kw)
target_achieved_strict = target_shortfall_kw <= 1e-9
comfort_scores = [x.comfort_score for x in optimized_decisions if x.comfort_score is not None]
mean_optimized_comfort_score = mean(comfort_scores) if comfort_scores else None
opted_out_building_ids = {x.building_id for x in occupants if x.opted_out}
opted_out_actual_reduction_kw = sum(x.estimated_reduction_kw for x in optimized_decisions
    if x.building_id in opted_out_building_ids)
counterfactual_context = replace(context, occupants=tuple(
    replace(x, opted_out=False) if x.building_id in opted_out_building_ids else x for x in occupants))
counterfactual_solver = RecordingSolver()
counterfactual_decisions = PlanningService(OptimizedStrategy(counterfactual_solver)).generate_plan(counterfactual_context)
opted_out_counterfactual_reduction_kw = sum(x.estimated_reduction_kw for x in counterfactual_decisions
    if x.building_id in opted_out_building_ids)
print('Reproducibility and run configuration')
print(json.dumps({
    'sample_dataset': str(SAMPLE_PATH.relative_to(PROJECT_ROOT)), 'sample_size_bytes': SAMPLE_PATH.stat().st_size,
    'seed': generation['seed'], 'dataset_parameters': parameters, 'dataset_counts': generation['counts'],
    'scenario': EXPERIMENT_SCENARIO, 'transformer_id': transformer.id,
    'transformer_capacity_kw': transformer.rated_capacity_kw,
    'planning_horizon_utc': [dr_event.start_time.isoformat(), dr_event.end_time.isoformat()],
    'horizon_intervals': int(dr_event.duration_minutes / 15),
    'strategy_pair': [comparison.first_strategy_name, comparison.second_strategy_name],
    'objective_weights': OBJECTIVE_WEIGHTS, 'target_reduction_kw': target_reduction_kw,
    'solver_status': solver_status, 'solver_feasible': recording_solver.last_outcome.feasible,
    'python': platform.python_version(), 'ortools': ortools.__version__, 'pandas': pd.__version__
}, indent=2))


Reproducibility and run configuration
{
  "sample_dataset": "data\\synthetic_demo.json",
  "sample_size_bytes": 264954,
  "seed": 2026,
  "dataset_parameters": {
    "appliances_per_building": 4,
    "building_count": 16,
    "interval_minutes": 15,
    "intervals_per_profile": 96,
    "occupants_per_building": 1,
    "transformer_count": 4
  },
  "dataset_counts": {
    "appliances": 64,
    "buildings": 16,
    "comfort_ranges": 16,
    "demand_response_events": 4,
    "load_profiles": 20,
    "occupants": 16,
    "tariffs": 4,
    "time_series_intervals": 1920,
    "transformers": 4
  },
  "scenario": "overloaded",
  "transformer_id": "TX-02",
  "transformer_capacity_kw": 31.960500000000003,
  "planning_horizon_utc": [
    "2026-07-16T17:00:00+00:00",
    "2026-07-16T20:00:00+00:00"
  ],
  "horizon_intervals": 12,
  "strategy_pair": [
    "baseline",
    "optimized"
  ],
  "objective_weights": {
    "peak_reduction": 0.7,
    "comfort": 0.3,
    "energy_cost": 0.25
  },
  "target_re

In [2]:
def show_kw(value): return f'{value:.4f}'
def show_pct(value): return f'{value:.2f}%'
rows = [
 {'Metric':'Uncontrolled measured event peak (kW)','Baseline':show_kw(uncontrolled_peak_kw),'Optimized':show_kw(uncontrolled_peak_kw),'Target':'Reference load','Result':'Shared measured profile before dispatch'},
 {'Metric':'Peak after strategy dispatch (kW)','Baseline':show_kw(baseline_metrics.optimized_peak_load_kw),'Optimized':show_kw(measured_optimized_peak_kw),'Target':f'<= {expected_optimized_peak_kw:.4f}','Result':f'{measured_optimized_peak_kw-expected_optimized_peak_kw:.5f} kW above target peak'},
 {'Metric':'Peak reduction from uncontrolled profile (kW)','Baseline':show_kw(baseline_metrics.peak_reduction_kw),'Optimized':show_kw(measured_reduction_kw),'Target':show_kw(target_reduction_kw),'Result':f'Strict shortfall {target_shortfall_kw:.5f} kW'},
 {'Metric':'Peak reduction from uncontrolled profile (%)','Baseline':show_pct(baseline_metrics.peak_reduction_pct),'Optimized':show_pct(optimized_metrics.peak_reduction_pct),'Target':show_pct(target_reduction_kw/uncontrolled_peak_kw*100),'Result':'Calculated against measured uncontrolled event peak'},
 {'Metric':'Target achieved (strict comparison)','Baseline':str(baseline_metrics.peak_reduction_kw>=target_reduction_kw),'Optimized':str(target_achieved_strict),'Target':'True','Result':'No; optimized result is short by the measured amount above'},
 {'Metric':'Mean comfort score for scored occupant decisions (0-1)','Baseline':'Not reported by baseline strategy','Optimized':f'{mean_optimized_comfort_score:.4f}' if mean_optimized_comfort_score is not None else 'Not reported','Target':'No numeric target','Result':f'Comfort impact vs ideal 1.0: {optimized_metrics.comfort_impact:.4f}'},
 {'Metric':'Distinct opted-out buildings','Baseline':str(len(opted_out_building_ids)),'Optimized':str(len(opted_out_building_ids)),'Target':'All protected by optimized strategy','Result':f'{optimized_metrics.opted_out_decision_count} opt-out decision returned'},
 {'Metric':'Reduction assigned to opted-out building(s) (kW)','Baseline':show_kw(sum(x.estimated_reduction_kw for x in baseline_decisions if x.building_id in opted_out_building_ids)),'Optimized':show_kw(opted_out_actual_reduction_kw),'Target':'0.0000', 'Result':f'Without the opt-out, counterfactual allocation is {opted_out_counterfactual_reduction_kw:.4f} kW'},
 {'Metric':'Planning decisions returned','Baseline':str(len(baseline_decisions)),'Optimized':str(len(optimized_decisions)),'Target':'One or more per building','Result':f'{sum(x.action.value=="defer_appliance" for x in optimized_decisions)} optimized appliance shifts'},
 {'Metric':'Solver status','Baseline':'Not applicable','Optimized':solver_status,'Target':'Feasible plan','Result':'Captured from underlying OR-Tools result'},
 {'Metric':'Estimated cost change vs no-dispatch (INR)','Baseline':f'{baseline_metrics.estimated_cost_difference:.2f}' if baseline_metrics.estimated_cost_difference is not None else 'Unavailable','Optimized':f'{optimized_metrics.estimated_cost_difference:.2f}' if optimized_metrics.estimated_cost_difference is not None else 'Unavailable','Target':'No cost target','Result':'Negative value means estimated cost reduction'},
]
results_table = pd.DataFrame(rows, columns=['Metric','Baseline','Optimized','Target','Result'])
results_table


,Metric,Baseline,Optimized,Target,Result
0,Uncontrolled measured event peak (kW),40.9750,40.9750,Reference load,Shared measured profile before dispatch
1,Peak after strategy dispatch (kW),28.7650,28.7683,<= 28.7650,0.00329 kW above target peak
2,Peak reduction from uncontrolled profile (kW),12.2100,12.2067,12.2100,Strict shortfall 0.00329 kW
3,Peak reduction from uncontrolled profile (%),29.80%,29.79%,29.80%,Calculated against measured uncontrolled event...
4,Target achieved (strict comparison),True,False,True,No; optimized result is short by the measured ...
5,Mean comfort score for scored occupant decisio...,Not reported by baseline strategy,0.6181,No numeric target,Comfort impact vs ideal 1.0: -0.3819
6,Distinct opted-out buildings,1,1,All protected by optimized strategy,1 opt-out decision returned
7,Reduction assigned to opted-out building(s) (kW),0.0000,0.0000,0.0000,"Without the opt-out, counterfactual allocation..."
8,Planning decisions returned,4,7,One or more per building,3 optimized appliance shifts
9,Solver status,Not applicable,optimal,Feasible plan,Captured from underlying OR-Tools result


## Baseline → Target → Measured Result

- **Uncontrolled measured peak:** 40.975 kW.
- **Baseline strategy post-dispatch peak:** 28.765 kW.
- **Target reduction:** 12.21 kW.
- **Expected peak at target:** 28.765 kW or lower.
- **Measured optimized peak:** 28.76829 kW.
- **Measured reduction:** 12.20671 kW (29.79%).
- **Target achieved (strict):** No; measured shortfall 0.00329 kW.
- **Error from target:** reduction minus target is -0.00329 kW; measured peak is 0.00329 kW above the target peak.

The result is reported as a strict target miss; the target is not rounded to claim success.


## Event load profile comparison

The plot uses the sample transformer's measured profile and adjusts it with returned decisions using the same event-slot adjustment logic as `PlanningAnalysisService`. It compares the uncontrolled profile, baseline dispatch, optimized dispatch, and target peak.


In [3]:
from html import escape
class InlineHTML:
    def __init__(self, markup): self.markup = markup
    def _repr_html_(self): return self.markup
    def __repr__(self): return 'Inline SVG event-load comparison'
def event_series_after_decisions(plan_context, decisions):
    # Reproduce event-slot profile adjustments used by PlanningAnalysisService.
    start = (plan_context.dr_event.start_time.hour*60+plan_context.dr_event.start_time.minute)//15
    count = max(1, int(plan_context.dr_event.duration_minutes/15))
    slots = tuple((start+i)%96 for i in range(count))
    profile = plan_context.transformer_load_profile
    before = {slot:profile.values_kw[slot%len(profile.values_kw)] for slot in slots}
    after = dict(before)
    reductions = sum(max(0.0,x.estimated_reduction_kw) for x in decisions if x.target_variable=='flexible_load_kw')
    for slot in slots: after[slot] = max(0.0,after[slot]-reductions)
    appliances_by_id = {x.id:x for x in plan_context.appliances}
    for decision in decisions:
        appliance=appliances_by_id.get(decision.target_variable or '')
        if appliance is None or decision.before_value is None or decision.after_value is None: continue
        old,new=int(decision.before_value),int(decision.after_value)
        for offset in range(appliance.duration_slots):
            old_slot,new_slot=(old+offset)%96,(new+offset)%96
            if old_slot in after: after[old_slot]=max(0.0,after[old_slot]-appliance.rated_power_kw)
            if new_slot in after: after[new_slot]+=appliance.rated_power_kw
    return slots,[before[x] for x in slots],[after[x] for x in slots]
event_slots,uncontrolled_series,baseline_series=event_series_after_decisions(context,baseline_decisions)
_,_,optimized_series=event_series_after_decisions(context,optimized_decisions)
assert abs(max(uncontrolled_series)-optimized_metrics.baseline_peak_load_kw)<1e-9
assert abs(max(optimized_series)-optimized_metrics.optimized_peak_load_kw)<1e-9
width,height=760,340; left,right,top,bottom=66,24,28,54
values=uncontrolled_series+baseline_series+optimized_series+[expected_optimized_peak_kw]
y_min,y_max=min(values)-1.0,max(values)+1.0
plot_width,plot_height=width-left-right,height-top-bottom
def px_x(i): return left+i*plot_width/max(1,len(event_slots)-1)
def px_y(v): return top+(y_max-v)*plot_height/(y_max-y_min)
def points(series): return ' '.join(f'{px_x(i):.1f},{px_y(v):.1f}' for i,v in enumerate(series))
colors={'Uncontrolled':'#64748b','Baseline strategy':'#0f766e','Optimized strategy':'#2563eb'}
svg=[f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width} {height}" role="img" aria-label="TX-02 measured and strategy load profiles">','<rect width="100%" height="100%" fill="white"/>']
for tick in range(5):
    value=y_min+tick*(y_max-y_min)/4; y=px_y(value)
    svg.append(f'<line x1="{left}" x2="{width-right}" y1="{y:.1f}" y2="{y:.1f}" stroke="#e2e8f0"/>')
    svg.append(f'<text x="{left-8}" y="{y+4:.1f}" text-anchor="end" font-size="11" fill="#334155">{value:.1f}</text>')
for i in range(0,len(event_slots),3):
    hour,quarter=divmod(event_slots[i],4)
    svg.append(f'<text x="{px_x(i):.1f}" y="{height-26}" text-anchor="middle" font-size="11" fill="#334155">{hour:02d}:{quarter*15:02d}</text>')
for name,series in [('Uncontrolled',uncontrolled_series),('Baseline strategy',baseline_series),('Optimized strategy',optimized_series)]:
    dash=' stroke-dasharray="5 4"' if name=='Uncontrolled' else ''
    svg.append(f'<polyline fill="none" stroke="{colors[name]}" stroke-width="2.5"{dash} points="{points(series)}"/>')
target_y=px_y(expected_optimized_peak_kw)
svg.append(f'<line x1="{left}" x2="{width-right}" y1="{target_y:.1f}" y2="{target_y:.1f}" stroke="#dc2626" stroke-dasharray="3 4"/>')
svg.append(f'<text x="{left}" y="18" font-size="13" font-weight="600" fill="#0f172a">{transformer.id} event load: {dr_event.duration_minutes/60:.0f} hours, 15-minute samples</text>')
for name,x in [('Uncontrolled',left),('Baseline',left+175),('Optimized',left+340),('Target peak',left+525)]:
    color=colors.get(name+' strategy',colors.get(name,'#dc2626'))
    if name=='Baseline': color=colors['Baseline strategy']
    if name=='Optimized': color=colors['Optimized strategy']
    dash=' stroke-dasharray="4 3"' if name in ('Uncontrolled','Target peak') else ''
    svg.append(f'<line x1="{x}" x2="{x+24}" y1="{height-8}" y2="{height-8}" stroke="{color}" stroke-width="2"{dash}/>')
    svg.append(f'<text x="{x+30}" y="{height-4}" font-size="10" fill="#334155">{escape(name)}</text>')
svg.append('</svg>')
load_profile_chart=InlineHTML(''.join(svg))
load_profile_chart


Inline SVG event-load comparison

## Optimized decisions and explanations

The comfort mean includes only optimized building decisions with a comfort score. Appliance shifts have no comfort score in the current model. `comfort_impact` is the scored average relative to an ideal score of 1.0.


In [4]:
explanation_service = ExplanationService()
decision_explanations = pd.DataFrame([
 {'Building':x.building_id,'Action':x.action.value,'Reduction (kW)':f'{x.estimated_reduction_kw:.4f}',
  'Comfort score':f'{x.comfort_score:.4f}' if x.comfort_score is not None else 'Not scored',
  'Explanation':explanation_service.explain(x)} for x in optimized_decisions
])
decision_explanations


,Building,Action,Reduction (kW),Comfort score,Explanation
0,B-005,reduce_setpoint,4.1267,0.4838,Building B-005: reduced flexible load by 4.1 k...
1,B-006,reduce_setpoint,4.0200,0.7767,Building B-006: reduced flexible load by 4.0 k...
2,B-007,opt_out_respected,0.0000,Not scored,Building B-007: no load was changed because th...
3,B-008,reduce_setpoint,4.0600,0.5940,Building B-008: reduced flexible load by 4.1 k...
4,B-005,defer_appliance,0.8653,Not scored,Building B-005: flexible appliance A-005-04 wa...
5,B-006,defer_appliance,0.8430,Not scored,Building B-006: flexible appliance A-006-04 wa...
6,B-008,defer_appliance,1.2179,Not scored,Building B-008: flexible appliance A-008-04 wa...


## Limitations and Error Analysis

The next cell executes the current infeasibility, missing-tariff, occupancy-input, and override paths. Its table distinguishes actual solver status tags from exceptions that escape the service/API boundary.


In [5]:
import logging
logging.getLogger('app.api.main').setLevel(logging.CRITICAL)
logging.getLogger('httpx').setLevel(logging.CRITICAL)
from fastapi.testclient import TestClient
from app.core.exceptions import MissingTariffDataError, OccupancySensorFailureError
from app.domain.entities.tariff import Tariff
from app.api.main import app

def run_optimized_case(test_context):
    test_solver=RecordingSolver()
    test_decisions=PlanningService(OptimizedStrategy(test_solver)).generate_plan(test_context)
    test_metrics=analysis_service.analyze(test_context,test_decisions)
    status=next((tag.split(':',1)[1] for x in test_decisions for tag in x.reasoning_tags
        if tag.startswith('SOLVER_STATUS:')),None)
    return test_solver,test_decisions,test_metrics,status

# Excessive target: existing solver returns infeasible tagged decisions.
huge_context=replace(context,dr_event=replace(dr_event,target_reduction_kw=999.0))
huge_solver,huge_decisions,huge_metrics,huge_status=run_optimized_case(huge_context)
# An overload that exceeds every available controllable load source.
low_capacity_transformer=replace(transformer,rated_capacity_kw=1.0)
low_capacity_context=replace(context,transformer=low_capacity_transformer)
capacity_solver,capacity_decisions,capacity_metrics,capacity_status=run_optimized_case(low_capacity_context)
# Missing prices are caught by analysis; physical planning still returns a plan.
missing_tariff_context=replace(context,tariff=Tariff('TAR-MISSING','Missing tariff','INR',{}))
tariff_solver,tariff_decisions,tariff_metrics,tariff_status=run_optimized_case(missing_tariff_context)
try:
    missing_tariff_context.tariff.get_rate(68)
except MissingTariffDataError as error:
    missing_tariff_exception=type(error).__name__
else:
    missing_tariff_exception='No exception'
occupancy_input_supported='occupancy_patterns' in context.__dataclass_fields__

# Exercise the real endpoint with a valid override body but no operator token.
unauthorized_payload={
 'dr_event':{'id':'DR-AUTH-CHECK','transformer_id':'TX-AUTH-CHECK','start_time':'2026-07-16T17:00:00Z','end_time':'2026-07-16T20:00:00Z','target_reduction_kw':1.0,'status':'active'},
 'transformer':{'id':'TX-AUTH-CHECK','name':'Authorization check transformer','rated_capacity_kw':10.0,'current_load_kw':12.0,'connected_building_ids':['B-AUTH-CHECK']},
 'buildings':[{'id':'B-AUTH-CHECK','name':'Authorization check building','transformer_id':'TX-AUTH-CHECK','building_type':'residential','floor_area_sqm':100.0,'fixed_load_kw':1.0,'flexible_load_kw':3.0,'occupant_ids':['O-AUTH-CHECK']}],
 'occupants':[{'id':'O-AUTH-CHECK','building_id':'B-AUTH-CHECK','display_name':'Synthetic occupant','comfort_range_id':'CR-AUTH-CHECK','opted_out':True,'allow_override':True}],
 'comfort_ranges':{'CR-AUTH-CHECK':{'id':'CR-AUTH-CHECK','variable':'temperature_c','min_value':19.0,'max_value':25.0,'preferred_value':22.0}},
 'emergency_override':{'operator_id':'experiment-check','justification':'Prevent transformer thermal damage'},
}
with TestClient(app) as test_client:
    unauthorized_response=test_client.post('/api/planning/emergency-override',json=unauthorized_payload)
unauthorized_error_type=unauthorized_response.json().get('error_type')
error_rows=[
 {'Scenario':'Demand target exceeds available flexible load','Expected behavior':'Return an infeasible plan status; do not fabricate reductions','Actual behavior':f'infeasible={huge_metrics.infeasible}; reductions={sum(x.estimated_reduction_kw for x in huge_decisions):.4f} kW','Exception/status':f'SOLVER_STATUS:{huge_status}; no exception raised','Limitation':'InsufficientReductionCapacityError is defined but this solver path reports a status tag instead.'},
 {'Scenario':'Required overload relief exceeds available controllable load','Expected behavior':'Mark the plan infeasible when available relief cannot meet transformer protection','Actual behavior':f'infeasible={capacity_metrics.infeasible}; rated capacity={low_capacity_transformer.rated_capacity_kw:.2f} kW','Exception/status':f'SOLVER_STATUS:{capacity_status}; no exception raised','Limitation':'The optimizer cannot relax the rating, comfort, or opt-out constraints.'},
 {'Scenario':'Missing tariff slot data','Expected behavior':'Continue physical planning and mark estimated cost unavailable','Actual behavior':f'infeasible={tariff_metrics.infeasible}; estimated_cost_difference={tariff_metrics.estimated_cost_difference}','Exception/status':f'{missing_tariff_exception} raised by Tariff.get_rate and caught by analysis','Limitation':'A cost comparison cannot be reported without rates for the analyzed slots.'},
 {'Scenario':'Occupancy/sensor data failure','Expected behavior':'Reject missing or invalid occupancy input if the planner validates it','Actual behavior':f'occupancy input in PlanningContext={occupancy_input_supported}; main plan ran','Exception/status':f'{OccupancySensorFailureError.__name__} is defined but not raised by current planning flow','Limitation':'Generated occupancy patterns are not passed into or validated by PlanningContext/API.'},
 {'Scenario':'Emergency override request without operator token','Expected behavior':'Reject the request before persisting a plan','Actual behavior':f'HTTP {unauthorized_response.status_code}; {unauthorized_error_type}','Exception/status':unauthorized_error_type,'Limitation':'This verifies authorization only; it does not measure emergency override effectiveness.'},
]
error_analysis_table=pd.DataFrame(error_rows,columns=['Scenario','Expected behavior','Actual behavior','Exception/status','Limitation'])
error_analysis_table


,Scenario,Expected behavior,Actual behavior,Exception/status,Limitation
0,Demand target exceeds available flexible load,Return an infeasible plan status; do not fabri...,infeasible=True; reductions=0.0000 kW,SOLVER_STATUS:infeasible_reduction_capacity; n...,InsufficientReductionCapacityError is defined ...
1,Required overload relief exceeds available con...,Mark the plan infeasible when available relief...,infeasible=True; rated capacity=1.00 kW,SOLVER_STATUS:infeasible_reduction_capacity; n...,"The optimizer cannot relax the rating, comfort..."
2,Missing tariff slot data,Continue physical planning and mark estimated ...,infeasible=False; estimated_cost_difference=None,MissingTariffDataError raised by Tariff.get_ra...,A cost comparison cannot be reported without r...
3,Occupancy/sensor data failure,Reject missing or invalid occupancy input if t...,occupancy input in PlanningContext=False; main...,OccupancySensorFailureError is defined but not...,Generated occupancy patterns are not passed in...
4,Emergency override request without operator token,Reject the request before persisting a plan,HTTP 403; UnauthorizedOverrideError,UnauthorizedOverrideError,This verifies authorization only; it does not ...


### Interpretation and limitations

- **Solver scale:** CP-SAT scales kW to integer hundredths and uses one search worker. This run covers one transformer, four buildings, and a 12-slot event; it does not establish performance at neighborhood-wide scale.
- **Infeasible scenarios:** An excessive target and an impossible transformer rating return decisions tagged `SOLVER_STATUS:infeasible_reduction_capacity`. This solver path does not raise the declared `InsufficientReductionCapacityError`; consumers must inspect decision tags or `infeasible` metrics.
- **Synthetic data:** Profiles, occupancy, tariffs, and transformer cases are generated examples, not calibrated telemetry or forecasts. Occupancy uses simple daily schedules; load traces use simplified HVAC/evening terms and random noise.
- **Occupancy assumptions:** Occupancy exists in the dataset but is absent from `PlanningContext` and the planning request. The current planner cannot detect missing/stale occupancy; `OccupancySensorFailureError` is defined but is not raised by this path.
- **Comfort model:** HVAC curtailment maps linearly to a setpoint change using a fixed degrees-C-per-kW factor. Scores are preference-distance proxies within hard bounds, not thermal dynamics or validated comfort measurements.
- **Opt-out interpretation:** TX-02 has 1 opted-out building(s); the optimizer assigned 0.0000 kW there. A counterfactual clearing the opt-out flag allocated 5.2701 kW to that building. This is an allocation counterfactual, not a claim that peak would improve by that amount; the actual plan reallocates actions to other buildings.
- **Peak and cost assumptions:** The analysis service applies total dispatch across each event slot and estimates cost from available TOU rates. It is not a meter replay or appliance-level time-varying load simulation.
- **Experiment size:** Results describe one seed, one transformer, four connected buildings, and a three-hour window. They do not support statistical generalization.
- **Override error:** The API returns HTTP 403 with `UnauthorizedOverrideError` when the token is missing. This tests authorization, not emergency override effectiveness.

The measured 0.00329 kW strict target shortfall is retained as an observed result; no solver behavior or target was altered to make the run pass.
